In [0]:
# ============================================================
# BRONZE TO SILVER: POS Transactions
# RetailPulse Data Pipeline
# ============================================================
# Reads raw POS CSV files from Bronze layer
# Produces two normalised Delta tables in Silver:
#   - retailpulse.silver.dim_transactions
#   - retailpulse.silver.fact_line_items
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DecimalType, TimestampType
)

# ------------------------------------------------------------
# CELL 1: Define Bronze schema explicitly
# We enforce schema on read — don't trust Bronze to have 
# correct types. Every field is declared explicitly.
# ------------------------------------------------------------

pos_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("line_item_id", StringType(), False),
    StructField("store_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DecimalType(10,2), False),
    StructField("discount_applied", DecimalType(10,2), True),
    StructField("line_total", DecimalType(10,2), False),
    StructField("transaction_total", DecimalType(10,2), False),
    StructField("payment_method", StringType(), False),
    StructField("transaction_time", StringType(), True),
])


In [0]:
# ------------------------------------------------------------
# CELL 2: Read Bronze POS data
# ------------------------------------------------------------

bronze_path = "abfss://bronze@retailpulsedatalake.dfs.core.windows.net/pos/"

df_bronze = spark.read \
    .option("header", "true") \
    .option("recursiveFileLookup", "true") \
    .schema(pos_schema) \
    .csv(bronze_path)

print(f"Bronze row count: {df_bronze.count()}")
display(df_bronze.limit(5))


Bronze row count: 2865


transaction_id,line_item_id,store_id,customer_id,product_id,product_name,quantity,unit_price,discount_applied,line_total,transaction_total,payment_method,transaction_time
7c93202a-474e-4cbb-8e5b-d41ea93f926e,7c93202a-474e-4cbb-8e5b-d41ea93f926e_LI_001,STORE_002,CUST_2184,SKU_001,Maize Meal 10kg,3,12.99,0.30,38.67,38.67,mobile_money,2026-05-25T00:33:45.037036
28b2c4cb-1592-43a3-aff2-d9d778eb6444,28b2c4cb-1592-43a3-aff2-d9d778eb6444_LI_001,STORE_003,CUST_7554,SKU_002,Cooking Oil 2L,7,8.49,0.96,58.47,115.54,card,2026-05-25T09:19:04.238959
28b2c4cb-1592-43a3-aff2-d9d778eb6444,28b2c4cb-1592-43a3-aff2-d9d778eb6444_LI_002,STORE_003,CUST_7554,SKU_003,Rice 5kg,3,9.99,1.27,28.70,115.54,card,2026-05-25T09:19:04.238959
28b2c4cb-1592-43a3-aff2-d9d778eb6444,28b2c4cb-1592-43a3-aff2-d9d778eb6444_LI_003,STORE_003,CUST_7554,SKU_006,Milk 1L,4,1.99,0.97,6.99,115.54,card,2026-05-25T09:19:04.238959
28b2c4cb-1592-43a3-aff2-d9d778eb6444,28b2c4cb-1592-43a3-aff2-d9d778eb6444_LI_004,STORE_003,CUST_7554,SKU_005,Bread Loaf,9,2.49,1.03,21.38,115.54,card,2026-05-25T09:19:04.238959


In [0]:
# ------------------------------------------------------------
# CELL 3: Clean and transform
# ------------------------------------------------------------

df_cleaned = df_bronze \
    .filter(F.col("transaction_id").isNotNull()) \
    .filter(F.col("line_item_id").isNotNull()) \
    .filter(F.col("quantity") > 0) \
    .filter(F.col("line_total") > 0) \
    .withColumn("discount_applied", 
        F.when(F.col("discount_applied").isNull(), 0)
        .otherwise(F.col("discount_applied"))
    ) \
    .withColumn("transaction_time_utc",
        F.to_utc_timestamp(
            F.to_timestamp(F.col("transaction_time")), 
            "Africa/Harare"
        )
    ) \
    .drop("transaction_time")

print(f"Cleaned row count: {df_cleaned.count()}")


Cleaned row count: 2865


In [0]:
# ------------------------------------------------------------
# CELL 4: Split into dim_transactions and fact_line_items
# ------------------------------------------------------------

# dim_transactions — one row per transaction (basket level)
dim_transactions = df_cleaned \
    .select(
        "transaction_id",
        "store_id", 
        "customer_id",
        "payment_method",
        "transaction_total",
        "transaction_time_utc"
    ) \
    .distinct()

# fact_line_items — one row per line item (product level)
fact_line_items = df_cleaned \
    .select(
        "line_item_id",
        "transaction_id",
        "product_id",
        "product_name",
        "quantity",
        "unit_price",
        "discount_applied",
        "line_total"
    )

print(f"dim_transactions row count: {dim_transactions.count()}")
print(f"fact_line_items row count: {fact_line_items.count()}")


dim_transactions row count: 960
fact_line_items row count: 2865


In [0]:
# ------------------------------------------------------------
# CELL 5: Write to Silver Delta tables
# ------------------------------------------------------------

# Write dim_transactions
dim_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailpulse.silver.dim_transactions")

# Write fact_line_items
fact_line_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailpulse.silver.fact_line_items")

print("Silver tables written successfully")


Silver tables written successfully


In [0]:
# ------------------------------------------------------------
# CELL 6: Verify Silver tables
# ------------------------------------------------------------

display(spark.sql("SELECT COUNT(*) as row_count FROM retailpulse.silver.dim_transactions"))
display(spark.sql("SELECT COUNT(*) as row_count FROM retailpulse.silver.fact_line_items"))
display(spark.sql("SELECT * FROM retailpulse.silver.dim_transactions LIMIT 5"))


row_count
960


row_count
2865


transaction_id,store_id,customer_id,payment_method,transaction_total,transaction_time_utc
49285b82-8867-4fa6-9716-9f7d9a5e4152,STORE_001,CUST_5648,card,165.86,2026-05-25T15:49:18.403Z
605b0c54-daea-4071-8d99-10bd144a929e,STORE_001,CUST_1258,cash,64.87,2026-05-25T10:54:15.267Z
a6ae37c8-0b6f-4cba-9cb7-0d5812c927ab,STORE_003,CUST_9295,cash,51.37,2026-05-25T16:19:12.435Z
7ca9682f-7d70-45b7-9596-ae1ac4063af0,STORE_002,CUST_9182,cash,194.17,2026-05-25T02:02:59.338Z
95a30c79-3ba6-4e5a-b891-7fb1d167d286,STORE_003,CUST_7651,card,58.39,2026-05-25T09:20:19.401Z
